# Post process data extracted using LLMs
1. Load data
2. Convert to correct format (numeric for numerical columns)
3. Eventually explode lists for geocoding?
4. Geocoding
5. Sanity checks

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import geopy as gpy
import time
import itertools
import regex as re
from matplotlib import pyplot as plt
from src.data import *
from src.plot_functions import *
from src.post_process_functions import *
from src.geocoding import *
from src.hazard_def import *
from src.impact_def import *
from src.sanity_checks import *



In [2]:
#load data (model)

res_savename = 'llm_response_impact_labelled_reports_test_multiprompt_16rep_meta-llama_llama-4-scout-17b-16e-instruct.csv'
response_df = pd.read_csv(DATA_OUT_LLMS+res_savename)

#load data (labelled)
#res_savename = "labelled_reports_impacts_all.csv"
#response_df = pd.read_csv(DATA_LABELLED+res_savename)


In [3]:
#load data (labelled)
#fnla = "labelled_8reports_impact_laura.csv"
#labelled_laura = pd.read_csv(DATA_LABELLED+fnla)
#fnlu = "labelled_reports_impacts_luca.csv"
#labelled_luca = pd.read_csv(DATA_LABELLED+fnlu)#.drop(["Unnamed: 0"],axis=1)
#
##reformat to be consistent
#labelled_laura["reportDate"] = pd.to_datetime(labelled_laura["reportDate"], dayfirst=True) #reformat date to be consistent
#labelled_luca["reportDate"] = pd.to_datetime(labelled_luca["reportDate"]) #reformat date to be consistent
#labelled_luca.rename({"annotation":"impactsAnnotation", "impactSubType":"impactSubtype"},inplace=True, axis=1)
#labelled_laura.rename({"annotation":"impactsAnnotation", "impactSubType":"impactSubtype"},inplace=True, axis=1)
#
#response_df = pd.concat([labelled_laura, labelled_luca]).reset_index(drop=True)
#res_savename = "labelled_reports_impacts_all.csv"
#response_df.to_csv(DATA_LABELLED+res_savename, index=False)


In [4]:
response_df[response_df["appealCode"]=="MDRUG050"]

,impactSubtype,impactValue,impactValueMin,impactValueMax,impactValuePrecision,impactUnit,valueAnnotation,country,location,locationAnnotation,...,endDay,dateAnnotation,hazards,hazardsAnnotation,appealCode,country_kw,reportDate,reportLink,disasterType,nathaz_text
134,Affected People,69283.0,NaN,NaN,exact,people,"['by august2024, the floods have affected69,28...",['Uganda'],"['Central region', 'Eastern region', 'Western ...","['targeted areas: central region, eastern regi...",...,NaN,"['by august2024, the floods have affected69,28...","['Flood', 'Other storm']",['this resulted in signicant impacts from epis...,MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['appeal: mdrug050total dref allocation: chf 4...
135,Injured People,9.0,NaN,NaN,exact,people,"['a hailstorm, impacting2,439 people, injuring...",['Uganda'],['Mbale'],"['in mbale district, bukasakya, and bungokho, ...",...,NaN,"['in mbale district, bukasakya, and bungokho, ...",['Other storm'],"['in mbale district, bukasakya, and bungokho, ...",MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['appeal: mdrug050total dref allocation: chf 4...
136,Injured People,166.0,NaN,NaN,exact,people,"['butaleja district, bukedi sub county, on apr...",['Uganda'],"['Butaleja district', 'Bukedi sub county']","['butaleja district, bukedi sub county, on apr...",...,NaN,"['butaleja district, bukedi sub county, on apr...","['Flood', 'Other storm']","['butaleja district, bukedi sub county, on apr...",MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['appeal: mdrug050total dref allocation: chf 4...
137,Displaced People,10366.0,NaN,NaN,exact,families,"['by august2024, the floods have affected69,28...",['Uganda'],"['Central region', 'Eastern region', 'Western ...","['targeted areas: central region, eastern regi...",...,NaN,"['by august2024, the floods have affected69,28...","['Flood', 'Other storm']","['in april2024, the eastern uganda-elgon regio...",MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['appeal: mdrug050total dref allocation: chf 4...
138,Displaced People,327.0,NaN,NaN,exact,people,"['in bweramule sub-county,714 households were ...",['Uganda'],['Bweramule sub-county'],"['in bweramule sub-county,714 households were ...",...,11.0,"['the rains intensied in the rst half of may, ...",['Flood'],"['the rains intensied in the rst half of may, ...",MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['appeal: mdrug050total dref allocation: chf 4...
139,Homeless People,905.0,NaN,NaN,exact,people,['this event also damaged six water facilities...,['Uganda'],['Sironko'],['this event also damaged six water facilities...,...,NaN,['this event also damaged six water facilities...,['Other storm'],['this event also damaged six water facilities...,MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['appeal: mdrug050total dref allocation: chf 4...
140,Homeless People,190.0,NaN,NaN,exact,people,['a landslide on the hilly slopes of sironko o...,['Uganda'],['Sironko'],['a landslide on the hilly slopes of sironko o...,...,NaN,['a landslide on the hilly slopes of sironko o...,['Mass movement'],['a landslide on the hilly slopes of sironko o...,MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['appeal: mdrug050total dref allocation: chf 4...
141,Homeless People,106.0,NaN,NaN,exact,people,"['in bududa district, buwali sub-county, affec...",['Uganda'],"['Bududa', 'Budwali sub-county']","['in bududa district, buwali sub-county, affec...",...,NaN,"['in bududa district, buwali sub-county, affec...","['Mass movement', 'Flood']","['in bududa district, buwali sub-county, affec...",MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['appeal: mdrug050total dref allocation: chf 4...
142,Missin

In [7]:
#get rid of nans
response_df = response_df.dropna(subset=["nathaz_text"]) if "nathaz_text" in response_df.columns else response_df

In [8]:
#convert numerical columns
num_cols = ["impactValue", "impactValueMin", "impactValueMax","startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]
list_cols = ["country","location", "hazards", "valueAnnotation", "locationAnnotation", "dateAnnotation", "hazardsAnnotation"]
response_df_proc = cp.deepcopy(response_df)
response_df_proc = format_output(response_df_proc, num_cols=num_cols, list_cols=list_cols)



In [9]:
#process impactValue

response_df_proc[["impactValue", "impactValueMin", "impactValueMax"]] = response_df_proc.apply(parse_impact_value_precision, axis=1)

/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:115: RuntimeWarning: All-NaN slice encountered
  min_value = np.nanmin(all_values)
/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:116: RuntimeWarning: All-NaN slice encountered
  max_value = np.nanmax(all_values)


In [10]:
response_df_proc[["impactValue", "impactValueMin", "impactValueMax"]]

,impactValue,impactValueMin,impactValueMax
0,7.6,NaN,NaN
1,300000.0,NaN,NaN
2,114.0,NaN,NaN
3,600000.0,NaN,NaN
4,532000.0,NaN,NaN
...,...,...,...
158,21000.0,NaN,NaN
159,NaN,NaN,NaN
160,0.5,NaN,NaN
161,NaN,NaN,NaN


In [11]:
#add iso3
response_df_proc["country_iso3"] = response_df_proc["country"].apply(country_name_to_iso3)
response_df_proc["country_iso3_kw"] = response_df_proc["country_kw"].apply(country_name_to_iso3) if "country_kw" in response_df_proc.columns else None

In [12]:
#format units
from spacy.lang.en import English
from spacy.lang.punctuation import TOKENIZER_PREFIXES, TOKENIZER_SUFFIXES, TOKENIZER_INFIXES
from spacy.lang.en import TOKENIZER_EXCEPTIONS
from spacy.tokenizer import Tokenizer
from spacy.util import compile_prefix_regex, compile_suffix_regex, compile_infix_regex

## Post process
1. Reclassify hazards
2. Reclassify impactSubtypes
3. Reclassify units

In [13]:
impactSubtype_list

['Affected People',
 'Injured People',
 'Displaced People',
 'Homeless People',
 'Missing People',
 'Human Deaths',
 'Human Health and Wellbeing',
 'Infected and Ill People',
 'Road Infrastructure',
 'Other Transportation Infrastructure',
 'Water, Sanitation, and Hygiene Infrastructure',
 'Healthcare Infrastructure',
 'IT and Communication Infrastructure',
 'Residential Buildings',
 'Informal settlements',
 'Education Infrastructure',
 'Power and Energy Production Infrastructure',
 'Agriculture Infrastructure',
 'Crop Production and Forestry',
 'Affected Livestock and Animals',
 'Other Economic and Livelihood Impacts',
 'Recreation, Tourism, and Culture',
 'Access to Healthcare',
 'Access to transport and Mobility',
 'Water Quality and Availability',
 'Access to Education',
 'Access to Power and Energy',
 'Access to Food',
 'Access to Water, Sanitation, and Hygiene',
 'Other Human Impacts',
 'Other Infrastructure Impacts',
 'Other Agricultural Impacts',
 'Other Service Access Impacts']

In [14]:
#reclassify impacType
impact_kw_reclass = {
    'Affected People': r"\bAffected People\b",
    'Injured People': r"\bInjured People\b",
    'Displaced People': r"\bDisplaced People\b",
    'Homeless People': r"\bHomeless People\b",
    'Missing People': r"\bMissing People\b",
    'Human Deaths': r"\bHuman Deaths\b",
    'Human Health and Wellbeing': r"\bHuman Health and Wellbeing\b",
    'Infected and Ill People': r"\bInfected and Ill People\b",

    'Road Infrastructure': r"\b(Road Infrastructure|road)s?\b",
    'Other Transportation Infrastructure': r"\bOther Transportation Infrastructure\b",
    'Water, Sanitation, and Hygiene Infrastructure': r"\bWater,?\s*Sanitation,?\s*and Hygiene Infrastructure\b",
    'Healthcare Infrastructure': r"\bHealthcare Infrastructure\b",
    'IT and Communication Infrastructure': r"\bIT and Communication Infrastructure\b",

    'Residential Buildings': r"\bResidential Buildings\b",
    'Informal settlements': r"\bInformal settlements\b",
    'Education Infrastructure': r"\bEducation Infrastructure\b",
    'Power and Energy Production Infrastructure': r"\bPower and Energy Production Infrastructure\b",
    'Agriculture Infrastructure': r"\bAgricultur(?:e|al)? Infras(?:tructure|tucture)\b",

    'Crop Production and Forestry': r"\bCrop Production and Forestry\b",
    'Affected Livestock and Animals': r"\bAffected Livestock and Animals\b",

    'Other Economic and Livelihood Impacts': r"\b(Other Economic(?: Activity)? (?:and|&) Livelihood (?:Production|Impact(?:s)?)|Economy and Market|Livelihood|employment|basic needs)\s*\b",
    'Recreation, Tourism, and Culture': r"\bRecreation, Tourism, and Culture\b",

    'Access to Healthcare': r"\bAccess to Healthcare\b",
    'Access to transport and Mobility': r"\bAccess to transport and Mobility\b",
    'Water Quality and Availability': r"\bWater Quality and Availability\b",
    'Access to Education': r"\bAccess to Education\b",
    'Access to Power and Energy': r"\bAccess to Power and Energy\b",
    'Access to Food': r"\bfood\b",
    'Access to Water, Sanitation, and Hygiene': r"\bAccess to Water,?\s*Sanitation,?\s*and Hygiene\b",

    'Other Infrastructure Impacts': r"\bOther Infrastructur(?:e|al)? Impacts?\b",
    'Other Human Impacts': r"\bOther Human.* Impacts?\b",
    'Other Environmental Impacts': r"\bOther Environmental.* Impacts?\b",
    'Other Service Access Impacts': r"\bOther Service Access.* Impacts?\b",
    'Other Agricultural Impacts': r"\bOther Agricultural.* Impacts?\b",
}
def reclassify_impact_subtype(extracted_data, allowed_impact_types, impact_kw_reclass):
    def reclass_impact_subtype(x):
        if x["impactSubtype"] in allowed_impact_types:
            return x["impactSubtype"]
        candidates = []
        for key, value in impact_kw_reclass.items():
            if re.search(value, x["impactSubtype"], re.IGNORECASE):
                candidates.append(key)
        if len(candidates) == 1:
            return candidates[0]
        else:
            return "Unknown"
    extracted_data["impactSubtype_orig"] = extracted_data["impactSubtype"]
    extracted_data["impactSubtype"] = extracted_data.apply(reclass_impact_subtype, axis=1)
    return extracted_data

response_df_proc = reclassify_impact_subtype(response_df_proc, impactSubtype_list, impact_kw_reclass)


In [15]:
response_df_proc[["impactSubtype", "impactSubtype_orig"]].where(response_df_proc["impactSubtype"]=="Unknown").dropna(how="all")

,impactSubtype,impactSubtype_orig


In [16]:
#reclassify hazard
hazard_kw_reclass = {
    'Drought': r"\bdrought.*|\bdry\s+spell.*",
    'Wildfire': r"\b(forest\s*fire|wild\s*fire|land\s*fire|bush\s*fire|wildfire|landfire|bushfire|fire)s?\b.*",
    'Earthquake': r"\b(earthquake|ground\s+movement|tsunami)s?\b.*",
    'Mass movement': r"\b(mass\s+movement|avalanche|land\s*slide|landslide|rockfall|sudden\s+subsidence|mudslide|rockslide)s?\b.*",
    'Volcanic activity': r"\b(volcanic|ash\s+fall|lava\s+flow|pyroclastic\s+flow|lahar)s?\b.*",
    'Flood': r"\b(flood|inundation|coastal\s+flood|flash\s+flood|riverine\s+flood|ice\s+jam\s+flood|heavy rain)s?\b.*",
    'Wave action': r"\b(wave|rogue\s+wave|seiche)\b.*",
    'Extreme cold temperature': r"\b(extreme\s+cold\s+temperature|cold\s+wave|coldwave|cold\s+spell|severe\s+winter\s+conditions)s?\b.*",
    'Extreme warm temperature': r"\b(extreme\s+warm\s+temperature|heat\s+wave|heatwave|heat\s+episode|(?:heat|hot)\s+spell|heat\s+stress)s?\b.*",
    'Tropical storm': r"\b(tropical\s+storm|typhoon|hurricane|cyclonic\s+storm)s?\b.*",
    'Other storm': r"\b(extra-?tropical\s+storm|winter\s*storm|storm\s+surge|superstorm|windstorm|snowstorm|blizzard|convective\s+storm|derecho|hail|lightning|tornado|thunderstorm)s?\b.*",
    'Epidemics': r"\b(cholera|dengue|outbreak|epidemic)s?\b.*",
    'Conflict': r"\b(conflict|war|terrorism|unrest)s?\b.*"
}

def reclassify_hazard(extracted_data, hazard_kw_reclass):
    def reclass_haz(x):
        corr_haz = cp.deepcopy(x["hazards"])
        if any([haz for haz in x["hazards"] if haz not in hazard_kw_reclass.keys()]):
            for i, haz in enumerate(x["hazards"]):
                if haz not in hazard_kw_reclass.keys():
                    candidates = [haz_corr for haz_corr in hazard_kw_reclass.keys() if re.search(hazard_kw_reclass[haz_corr], haz, re.IGNORECASE)]
                    if len(candidates) == 1:
                        corr_haz[i] = candidates[0]
                    else:
                        corr_haz[i] = "Unknown"
        return corr_haz
    extracted_data["hazards_reclass"] = extracted_data.apply(reclass_haz, axis=1)
    return extracted_data

response_df_proc = reclassify_hazard(response_df_proc, hazard_kw_reclass)
explode_lists(response_df_proc).hazards_reclass.value_counts()

/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:89: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  repeat_counts = df[list_columns].applymap(len).max(axis=1)


hazards_reclass
Flood             574
Mass movement     226
Other storm        86
Epidemics          26
Tropical storm     13
Name: count, dtype: int64

In [17]:
wrong_haz = explode_lists(response_df_proc).copy()
wrong_haz = wrong_haz[wrong_haz["hazards_reclass"] == "Unknown"]
wrong_haz["hazards"]

/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:89: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  repeat_counts = df[list_columns].applymap(len).max(axis=1)


Series([], Name: hazards, dtype: object)

In [18]:
impactSubtype_list

['Affected People',
 'Injured People',
 'Displaced People',
 'Homeless People',
 'Missing People',
 'Human Deaths',
 'Human Health and Wellbeing',
 'Infected and Ill People',
 'Road Infrastructure',
 'Other Transportation Infrastructure',
 'Water, Sanitation, and Hygiene Infrastructure',
 'Healthcare Infrastructure',
 'IT and Communication Infrastructure',
 'Residential Buildings',
 'Informal settlements',
 'Education Infrastructure',
 'Power and Energy Production Infrastructure',
 'Agriculture Infrastructure',
 'Crop Production and Forestry',
 'Affected Livestock and Animals',
 'Other Economic and Livelihood Impacts',
 'Recreation, Tourism, and Culture',
 'Access to Healthcare',
 'Access to transport and Mobility',
 'Water Quality and Availability',
 'Access to Education',
 'Access to Power and Energy',
 'Access to Food',
 'Access to Water, Sanitation, and Hygiene',
 'Other Human Impacts',
 'Other Infrastructure Impacts',
 'Other Agricultural Impacts',
 'Other Service Access Impacts']

### Units reclassification
1. Standardize units i.e. metric units to SI or common units.
2. Determine unit typology (e.g. distance, surface, weight, percent)
3. Convert non-metric units e.g. households to people, USD to CHF
4. Standardize non-metric units i.e. person, children to people, hospitals to health facilities

TO DOS
1. Handle deaths so that they dont go in other categories
2. Handle targeted people

In [19]:
#reclassify units
unit_converter = {"families" : (3, "people"),
                  "households": (3, "people"),
                  "village": (1000, "people"),
                  "communities": (100, "people"),
                  "USD": (1, "CHF"),
                  "$" : (1, "CHF"),
                  "EUR": (1, "CHF"),
                  "€": (1, "CHF"),
                  }

unit_type_kw_reclass = {#regex actually here should not be necessary due to standardization
                        'km' : r"\b(kilometer|kilometre|km)s?(?!\s*(\*\*\s*2|\^2|²|square|squared|2))",
                        'km**2' : r"\b(kilometer|kilometre|km)s?\s?(\*\*\s*2|\*\*2|\^2|²|square|squared|2)",
                        'kg' : r"(kg.*|.*kilogram.*)",
                        'm**3' : r"\b(meter|metre|m)s?\s?(\*\*\*\s*3|\*\*3|\^3|³|cube|cubic|3)",
                        '%' : r"(%|perc.*)",
}
unit_kw_reclass ={
                        'people': r"people.*|person.*|women.*|men.*|child.*|children.*|adult.*|adults.*|elder.*|elderly.*|infant.*|infants.*|individual.",
                        'roads' : r"road.*|route.*|bridge.*|highway.*|motorway.*",#r"(?<!kilometer|kilometre|km).*(road.*|route.*|.*bridge.*|.*highway.*|.*motorway.*)",
                        'transportation facilities' : r"rail.*|train track.*|airport.*|\scar.*|railway.*|train.*|bus.*|taxi.*|taxicab.*|truck.*",
                        'water, sanitation and hygiene facilities' : r"water.*|sanitation.*|hygiene.*|latrine.*|well.*|tap.*|reservoir.*|aqueduct.*",
                        'healthcare facilities' : r"health|hospital.*|clinic.*|maternity.*|medical",
                        'IT and communication facilities' : r"communication.*|radio.*|tv.*|cell tower.*|antenna.*",
                        'homes' : r"residential.*|residence.|hous.*|home.*|building.*",
                        'education facilities' : r"education.*|school.*|university.*|college.*",
                        'crop production and forestry' : r"crop.*|field.*|forest.*|tree.*|banana.*|coffee.*|cocoa.*|cotton.*|maize.*|rice.*|sorghum.*|soybean.*|sugar.*|tobacco.*|wheat.*",
                        'agricultural facilities' : r"irrigation.*|barn.*|farm.*",
                        'affected animals' : r"livestock.*|animal.*|fish.*|cow.*|sheep.*|poult.*|cattle.*|goat.*|pig.*|chick.*|horse.*|heads?",
                        'informal settlements' : r"camp.?|tent.?|refuge.?|settlement.?"
                         }
default_subtype_unit = {
 'Affected People': "people",
 'Injured People': "people",
 'Displaced People': "people",
 'Homeless People': "people",
 'Missing People': "people",
 'Human Deaths': "people",
 'Residential Buildings': "homes",
 'Informal settlements': "undefined informal settlements",
 'Education Infrastructure': "schools",
 'Human Health and Wellbeing' : "unknown",
 'Infected and Ill People': "people",
 'Road Infrastructure' : "roads",
 'Other Transportation Infrastructure' : "undefined other transportation infrastructure",
 'Water, Sanitation, and Hygiene Infrastructure': "undefined WASH facilities",
 'Healthcare Infrastructure': "undefined healthcare facilities",
 'IT and Communication Infrastructure': "undefined IT and communication facilities",
 'Residential Buildings': "houses",
 'Informal settlements': "undefined informal settlements",
 'Education Infrastructure': "schools",
 'Power and Energy Production Infrastructure' : "undefined power and energy production infrastructure facilities",
 'Agriculture Infrastructure': "undefined agricultural facilities",
 'Crop Production and Forestry': "undefined crop production and forestry",
 'Affected Livestock and Animals': "undefined affected animals",
 'Other Economic and Livelihood Impacts': "CHF",
 #'Water Quality and Availability':
 'Recreation, Tourism, and Culture' : "unknown",
 'Access to Healthcare': "people",
 'Access to transport and Mobility': "people",
 'Water Quality and Availability' : "unknown",
 'Access to Education':"people",
 'Access to Power and Energy':"people",
 'Access to Food':"people",
 'Access to Water, Sanitation, and Hygiene':"people",
 'Other Human Impacts': "unknown",
 'Other Infrastructure Impacts': "unknown",
 'Other Agricultural Impacts': "unknown",
 'Other Service Access Impacts': "people"
}

def convert_unit(extracted_data, unit_converter):
    """Convert units that can be converted e.g. families => people"""
    def convert(x):
        unit = x['impactUnit']
        if not isinstance(unit, str):
            return x  # skip if unit is None or not a string

        unit = unit.strip()
        if unit == "":
            return x

        for old_unit, (conv_fact, new_unit) in unit_converter.items():
            try:
                if unit == old_unit:
                    x["impactValue"] = float(x["impactValue"]) #force conversion to float
                    x["impactValue"] = conv_fact*x["impactValue"]
                    x["impactUnit"] = new_unit
            except Exception as e:
                print(f"Skipping unit conversion for row due to error: {e}")
                continue
        return x
    extracted_data = extracted_data.apply(convert, axis=1)
    return extracted_data


def assign_unit_type(extracted_data, unit_type_kw_reclass):
    """Detect if dimension of unit can be identified e.g. length, mass,...
       Default to "other"
    """
    def assign_type(x):
        unit = str(x["impactUnit"]).lower() #ensure unit is string
        candidates = [unit_type for unit_type in unit_type_kw_reclass.keys() if re.search(unit_type_kw_reclass[unit_type], unit, re.IGNORECASE)]
        if len(candidates) == 1:
            return candidates[0]
        elif len(candidates) == 0:
            return "other"
        else:
            return "multiple"
    extracted_data["unit_type"] = extracted_data.apply(assign_type, axis=1)
    return extracted_data

def reclassify_units(extracted_data, unit_kw_reclass, default_subtype_unit):
    def reclass_units(x):
        unit = str(x["impactUnit"]).lower() #ensure unit is string
        unit_type = x['unit_type']
        unit_prefix = f"{unit_type} of " if unit_type != "other" else ""
        candidates = [unit_corr for unit_corr in unit_kw_reclass.keys() if re.search(unit_kw_reclass[unit_corr], unit, re.IGNORECASE)]
        if len(candidates) == 1:
            return unit_prefix+candidates[0]
        else:
            #no unit identified, infer unit from category
            inferred_unit = default_subtype_unit[x["impactSubtype"]] if x["impactSubtype"] != "Unknown" else unit
            return unit_prefix+inferred_unit
    extracted_data["impactUnit"] = extracted_data.apply(reclass_units, axis=1)
    return extracted_data

from src.text_processing_functions import *
def standardize_units(value, unit):
    """Standardize units to a common baseline in text"""

    ureg = UnitRegistry()
    #print(f"{value} {unit}")
    identified_units = []
    identified_patterns = []
    for target_unit, unit_patterns in std_unit_kw_reclass.items():
        for pattern in unit_patterns:
            if re.search(pattern, unit, re.IGNORECASE):
                identified_units.append(target_unit)
                identified_patterns.append(pattern)
    #matched = [(target_unit, unit_patterns) for target_unit, unit_patterns in std_unit_kw_reclass.items() if np.any([re.search(pattern, unit, re.IGNORECASE) for pattern in unit_patterns])]
    if len(identified_units) == 0:
        return pd.Series({"impactValue": value, "impactUnit": unit})
    elif len(identified_units) > 1:
        raise ValueError(f"Multiple potential units found for token: {unit}")
    identified_unit = identified_units[0]
    identified_pattern = identified_patterns[0]
    si_unit = unit_mapping[identified_unit]
    # Perform conversion
    quantity = float(value) * ureg(identified_unit)
    converted_quantity = quantity.to(si_unit)
    converted_value = converted_quantity.magnitude
    converted_unit = re.sub(identified_pattern, si_unit, unit)

    return pd.Series({"impactValue": converted_value, "impactUnit": converted_unit})

def standardize_value_units(response_df):
    def join_value_units(x):
        return str(x["impactValue"]) +  "," + str(x["impactUnit"])
    def split_value_units(x):
        return x["value_unit"].split(",")
    def apply_std_units(x):
        return standardize_units(str(x["impactValue"]), str(x["impactUnit"]))
    #response_df["value_unit"] = response_df.apply(join_value_units, axis=1)
    #response_df["value_unit"] = response_df.apply(split_value_units, axis=1)
    response_df[["impactValue", "impactUnit"]]  = response_df.apply(apply_std_units, axis=1)
    return response_df



In [20]:
unit_type = "kg"
unit = "kg of crops"
[unit_corr for unit_corr in unit_kw_reclass.keys() if re.search(unit_kw_reclass[unit_corr], unit, re.IGNORECASE)]


['crop production and forestry']

In [21]:
test_df = pd.DataFrame({
        "impactSubtype": ["Affected People", "Crop Production and Forestry", "Crop Production and Forestry"],
        "impactValue": [1000,1000,100],
        "impactUnit": ["families", "kg of crops","hectares of crops"]})
test_df = standardize_value_units(test_df)
test_df = convert_unit(test_df, unit_converter)
test_df = assign_unit_type(test_df, unit_type_kw_reclass)
test_df = reclassify_units(test_df, unit_kw_reclass, default_subtype_unit)
test_df

,impactSubtype,impactValue,impactUnit,unit_type
0,Affected People,3000.0,people,other
1,Crop Production and Forestry,1000.0,kg of crop production and forestry,kg
2,Crop Production and Forestry,1.0,km**2 of crop production and forestry,km**2


In [22]:
#keep orig unit and value for comparison
response_df_proc["impactValueOrig"] = response_df_proc["impactValue"]
response_df_proc["impactUnitOrig"] = response_df_proc["impactUnit"]
response_df_proc = standardize_value_units(response_df_proc)
response_df_proc = convert_unit(response_df_proc, unit_converter)
response_df_proc = assign_unit_type(response_df_proc, unit_type_kw_reclass)
response_df_proc = reclassify_units(response_df_proc, unit_kw_reclass, default_subtype_unit)

In [25]:
response_df_proc[["impactSubtype","impactValueOrig", "impactValue", "impactUnitOrig", "impactUnit","valueAnnotation"]]

,impactSubtype,impactValueOrig,impactValue,impactUnitOrig,impactUnit,valueAnnotation
0,Affected People,7.6,7.6,million people,people,[more than7.6 million people were affected in2...
1,Displaced People,300000.0,300000.0,people,people,"[over300,000 people displaced]"
2,Human Deaths,114.0,114.0,people,people,[114 people dead]
3,Residential Buildings,600000.0,600000.0,houses,homes,"[approximately600,000 houses damaged]"
4,Crop Production and Forestry,532000.0,5320.0,hectares of crops,km**2 of crop production and forestry,"[about532,000 hectares of crops were destroyed]"
...,...,...,...,...,...,...
158,Human Health and Wellbeing,21000.0,21000.0,reported cases,unknown,[the country continues to recover from a chole...
159,Crop Production and Forestry,NaN,nan,minimal,undefined crop production and forestry,[although most of the surveyed households repo...
160,Affected Livestock and Animals,0.5,0.5,half of surveyed households,homes,"[likewise, livestock and fisheries are equally..."
161,Water Quality and Availability,NaN,nan,NaN,unknown,[the decreased access to water has also led to...


In [26]:
# save
savename = "post_processed_" + res_savename
response_df_proc.to_csv(DATA_OUT_PROC + savename, index=False)

In [28]:
response_df_proc[["impactValue", "impactUnit", "impactValueOrig", "impactUnitOrig", "valueAnnotation"]]

,impactValue,impactUnit,impactValueOrig,impactUnitOrig,valueAnnotation
0,7.6,people,7.6,million people,[more than7.6 million people were affected in2...
1,300000.0,people,300000.0,people,"[over300,000 people displaced]"
2,114.0,people,114.0,people,[114 people dead]
3,600000.0,homes,600000.0,houses,"[approximately600,000 houses damaged]"
4,5320.0,km**2 of crop production and forestry,532000.0,hectares of crops,"[about532,000 hectares of crops were destroyed]"
...,...,...,...,...,...
158,21000.0,unknown,21000.0,reported cases,[the country continues to recover from a chole...
159,nan,undefined crop production and forestry,NaN,minimal,[although most of the surveyed households repo...
160,0.5,homes,0.5,half of surveyed households,"[likewise, livestock and fisheries are equally..."
161,nan,unknown,NaN,NaN,[the decreased access to water has also led to...
